# Baseline review

Consultation only — **nothing here runs a model**. Every file was computed in the ADR-0005 baseline
sessions and lives in `artifacts/`. All files are level-matched with a common headroom gain, so no
comparison is decided by loudness and nothing clips.

Two comparison sets:

- **AN-2** (7.6 min, mono, brick-walled at its 15750 Hz knee before A2SB): before / Apollo /
  A2SB 1-split / A2SB 2-split. Apollo ran on the uncut mono original — its proper input.
- **codec_wav** (6 s, mono): input / Apollo on the original / A2SB 2-split ensemble on the 4 kHz
  brick wall / A2SB 1-split, kept as a labeled example of the flat-shelf artifact.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from grooveback import audio as ga

REPO = Path("..")
LISTEN = REPO / "artifacts" / "listen"

SETS = {
    "an2": {
        "before":       LISTEN / "mono_an2_before.wav",
        "apollo":       LISTEN / "mono_an2_apollo.wav",
        "a2sb 1-split": LISTEN / "mono_an2_a2sb_1split.wav",
        "a2sb 2-split": LISTEN / "mono_an2_a2sb_2split.wav",
    },
    "codec": {
        "input":         LISTEN / "mono_codec_input.wav",
        "apollo":        LISTEN / "mono_codec_apollo.wav",
        "a2sb ensemble": LISTEN / "mono_codec_a2sb_ensemble.wav",
        "a2sb 1-split (flat-shelf artifact)": LISTEN / "mono_codec_a2sb.wav",
    },
}

audio = {}
for set_name, files in SETS.items():
    audio[set_name] = {}
    for label, path in files.items():
        signal, sr = ga.load(path)
        audio[set_name][label] = (signal, sr)
        print(f"{set_name:6} {label:36} {signal.shape[1]/sr:6.1f}s  "
              f"{ga.loudness(signal, sr):+7.2f} LUFS  peak {ga.peak_dbfs(signal):+6.2f} dBFS")

## Listen

Streamed from disk, not embedded — full-length playback costs nothing. Headphones or monitors.

In [ ]:
for set_name, files in SETS.items():
    print(f"=== {set_name}")
    for label, path in files.items():
        print(label)
        display(Audio(url=f"/files/{path.relative_to(REPO)}"))

## Spectrograms, back to back

Calibrated dBFS, shared −120…0 scale — trustable against Spek. One row per method.

In [ ]:
def spectrogram_grid(set_name, floor_db=-120):
    items = audio[set_name]
    fig, axes = plt.subplots(len(items), 1, figsize=(16, 3.2 * len(items)),
                             constrained_layout=True, sharex=True)
    for ax, (label, (signal, sr)) in zip(np.atleast_1d(axes), items.items()):
        spec = ga.spectrogram_db(signal)
        im = ax.imshow(spec, origin="lower", aspect="auto",
                       extent=[0, signal.shape[1] / sr, 0, sr / 2000],
                       vmin=floor_db, vmax=0, cmap="magma")
        ax.set_ylabel("kHz")
        ax.set_title(label, fontsize=10, loc="left")
    np.atleast_1d(axes)[-1].set_xlabel("time (s)")
    fig.colorbar(im, ax=axes, label="dBFS", shrink=0.6)
    plt.show()


for set_name in SETS:
    spectrogram_grid(set_name)

## Difference vs before

Red is energy the method added, blue is energy it removed. The honest plot — a method that "does
nothing below the cutoff" should be white there.

In [ ]:
def difference_grid(set_name):
    items = dict(audio[set_name])
    ref_label = next(iter(items))
    ref, sr = items.pop(ref_label)
    ref_spec = ga.spectrogram_db(ref)
    fig, axes = plt.subplots(len(items), 1, figsize=(16, 3.2 * len(items)),
                             constrained_layout=True, sharex=True)
    for ax, (label, (signal, _)) in zip(np.atleast_1d(axes), items.items()):
        spec = ga.spectrogram_db(signal)
        n = min(spec.shape[1], ref_spec.shape[1])
        delta = spec[:, :n] - ref_spec[:, :n]
        limit = float(np.percentile(np.abs(delta), 99))
        im = ax.imshow(delta, origin="lower", aspect="auto",
                       extent=[0, signal.shape[1] / sr, 0, sr / 2000],
                       vmin=-limit, vmax=limit, cmap="RdBu_r")
        ax.set_ylabel("kHz")
        ax.set_title(f"{label}  −  {ref_label}", fontsize=10, loc="left")
        fig.colorbar(im, ax=ax, label="dB")
    np.atleast_1d(axes)[-1].set_xlabel("time (s)")
    plt.show()


for set_name in SETS:
    difference_grid(set_name)

## Band energies

The numbers behind the colours.

In [ ]:
BANDS = [(14000, 15000), (15000, 16000), (16000, 17000), (17000, 18000),
         (18000, 19000), (19000, 20000), (20000, 21000), (21000, 22050)]

for set_name, items in audio.items():
    labels = list(items)
    print(f"=== {set_name}")
    print(f"{'band':>13} " + " ".join(f"{label[:12]:>12}" for label in labels))
    for lo, hi in BANDS:
        row = " ".join(
            f"{ga.band_energy_db(items[label][0], items[label][1], lo, hi):+12.1f}"
            for label in labels
        )
        print(f"{lo/1000:5.1f}-{hi/1000:4.1f}k {row}")
    print()

## Zoom

Close inspection of any window — seams, transients, the crackle hunt. `zoom("an2", 120, 10)`
re-plots all methods over that window at full spectral resolution.

In [ ]:
def zoom(set_name, start_seconds, length_seconds, floor_db=-120):
    items = audio[set_name]
    fig, axes = plt.subplots(len(items), 1, figsize=(16, 3.0 * len(items)),
                             constrained_layout=True, sharex=True)
    for ax, (label, (signal, sr)) in zip(np.atleast_1d(axes), items.items()):
        a = int(start_seconds * sr)
        b = min(a + int(length_seconds * sr), signal.shape[1])
        spec = ga.spectrogram_db(signal[:, a:b], max_frames=100000)
        im = ax.imshow(spec, origin="lower", aspect="auto",
                       extent=[start_seconds, b / sr, 0, sr / 2000],
                       vmin=floor_db, vmax=0, cmap="magma")
        ax.set_ylabel("kHz")
        ax.set_title(label, fontsize=10, loc="left")
    np.atleast_1d(axes)[-1].set_xlabel("time (s)")
    plt.show()


zoom("an2", 120, 10)